In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from dotenv import load_dotenv

from langgraph.graph.message import add_messages

from langgraph.checkpoint.memory import MemorySaver

/home/sahil/Desktop/LANGGRAPH/myenv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
load_dotenv()

llm = ChatOpenAI()

In [3]:
class ChatState(TypedDict):
    
    messages: Annotated[list[BaseMessage], add_messages]

In [4]:
def chat_node(state: ChatState):
    #prompt
    messages = state['messages']

    response = llm.invoke(messages)

    return {'messages': [response]}

In [5]:
checkpointer = MemorySaver()

graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chat_bot = graph.compile(checkpointer=checkpointer)

In [ ]:
thread_id =  '1'

while True:
    try:
        user_message = input("User: ")
    except EOFError:
        break  # stops notebook crash

    if not user_message:
        continue

    user_message = user_message.strip()

    if user_message.lower() in ["exit", "quit", "bye"]:
        print("Bot: Exiting...")
        break

    config = {'configurable': {'thread_id': thread_id}}

    response = chat_bot.invoke({
        'messages': [HumanMessage(content=user_message)]}, config=config)

    print(f"Bot: {response['messages'][-1].content}")